In [ ]:
!pip install pandas
!pip install ipython-sql prettytable

import prettytable

prettytable.DEFAULT = 'DEFAULT'

In [ ]:
import csv, sqlite3

con = sqlite3.connect("Chicago.db")
cur = con.cursor()

In [ ]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
import pandas

df = pandas.read_csv("Census_Data.csv")
df.to_sql("CENSUS_DATA", con, if_exists='replace', index=False, method="multi")

df1 = pandas.read_csv("Chicago_Public_Schools.csv")
df1.to_sql("CHICAGO_PUBLIC_SCHOOLS", con, if_exists='replace', index=False, method="multi")

df2 = pandas.read_csv("Chicago_Crime_Data.csv")
df2.to_sql("CHICAGO_CRIME_DATA", con, if_exists='replace', index=False, method="multi")

533

In [ ]:
%sql sqlite:///Chicago.db

In [ ]:
%sql select count(*) from CENSUS_DATA

 * sqlite:///Chicago.db
Done.


count(*)
78


In [ ]:
# 1. Find the total number of crimes recorded in the CRIME table.
%sql select count(*) from CHICAGO_CRIME_DATA

 * sqlite:///Chicago.db
Done.


count(*)
533


In [ ]:
# Checking columns names
%%sql PRAGMA table_info(CENSUS_DATA)

 * sqlite:///Chicago.db
Done.


cid,name,type,notnull,dflt_value,pk
0,COMMUNITY_AREA_NUMBER,REAL,0,None,0
1,COMMUNITY_AREA_NAME,TEXT,0,None,0
2,PERCENT OF HOUSING CROWDED,REAL,0,None,0
3,PERCENT HOUSEHOLDS BELOW POVERTY,REAL,0,None,0
4,PERCENT AGED 16+ UNEMPLOYED,REAL,0,None,0
5,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,REAL,0,None,0
6,PERCENT AGED UNDER 18 OR OVER 64,REAL,0,None,0
7,PER_CAPITA_INCOME,INTEGER,0,None,0
8,HARDSHIP_INDEX,REAL,0,None,0


In [ ]:
# 2. List community areas with per capita income less than 11000
%%sql
 select COMMUNITY_AREA_NAME
 from CENSUS_DATA
 where `PER_CAPITA_INCOME `< 11000;

 * sqlite:///Chicago.db
Done.


COMMUNITY_AREA_NAME
West Garfield Park
South Lawndale
Fuller Park
Riverdale


In [ ]:
# 3. List all case numbers for crimes involving minors
%sql select CASE_NUMBER from CHICAGO_CRIME_DATA where description like '%MINOR%'

 * sqlite:///Chicago.db
Done.


CASE_NUMBER
HL266884
HK238408


In [ ]:
# 4. List all kidnapping crimes involving a child?(children are not considered minors
# for the purposes of crime analysis)
%sql select * from CHICAGO_CRIME_DATA where PRIMARY_TYPE='KIDNAPPING' and DESCRIPTION like '%CHILD%'

 * sqlite:///Chicago.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,UPDATEDON,LATITUDE,LONGITUDE,LOCATION
5276766,HN144152,01/26/2007 10:05:00 AM,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,02/28/2018 03:56:25 PM,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


In [ ]:
# 5. What kind of crimes were recorded at schools?
%sql select DISTINCT PRIMARY_TYPE from CHICAGO_CRIME_DATA where LOCATION_DESCRIPTION like '%SCHOOL%'

 * sqlite:///Chicago.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


In [ ]:
# 6. List the average safety score for all types of schools.
%%sql
select `Elementary, Middle, or High School`, AVG(SAFETY_SCORE)
from CHICAGO_PUBLIC_SCHOOLS
group by `Elementary, Middle, or High School`

 * sqlite:///Chicago.db
Done.


"Elementary, Middle, or High School",AVG(SAFETY_SCORE)
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


In [ ]:
# 7. List 5 community areas with highest % of households below poverty line.
%%sql
select COMMUNITY_AREA_NAME
from CENSUS_DATA
order by `PERCENT HOUSEHOLDS BELOW POVERTY` desc
limit 5

 * sqlite:///Chicago.db
Done.


COMMUNITY_AREA_NAME
Riverdale
Fuller Park
Englewood
North Lawndale
East Garfield Park


In [ ]:
# 8. Which community area(number) is most crime prone?
%%sql
select cd.COMMUNITY_AREA_NUMBER
from CENSUS_DATA cd, CHICAGO_CRIME_DATA ccd
where cd.COMMUNITY_AREA_NUMBER=ccd.COMMUNITY_AREA_NUMBER
group by cd.COMMUNITY_AREA_NUMBER
order by count(ccd.ID) desc
limit 1

 * sqlite:///Chicago.db
Done.


COMMUNITY_AREA_NUMBER
25.0


In [ ]:
# 9. Use a sub-query to find the name of the community area with highest hardship
%%sql
select COMMUNITY_AREA_NAME
from CENSUS_DATA
where HARDSHIP_INDEX = (select MAX(HARDSHIP_INDEX) from CENSUS_DATA)

 * sqlite:///Chicago.db
Done.


COMMUNITY_AREA_NAME
Riverdale


In [ ]:
# 10. Use a sub-query to determine the Community Area Name with most number
# of crimes?
%%sql
select COMMUNITY_AREA_NAME, NUMBER_CRIMES
from (
    	 select cd.COMMUNITY_AREA_NAME, count(ccd.ID) AS NUMBER_CRIMES
       from CENSUS_DATA cd, CHICAGO_CRIME_DATA ccd
            where cd.COMMUNITY_AREA_NUMBER = ccd.COMMUNITY_AREA_NUMBER
            group by cd.COMMUNITY_AREA_NAME
            order by NUMBER_CRIMES desc
    )
LIMIT 1;

 * sqlite:///Chicago.db
Done.


COMMUNITY_AREA_NAME,NUMBER_CRIMES
Austin,43


In [ ]:
# 11. How many Elementary Schools are in the dataset?
%%sql
select COUNT(*)
from CHICAGO_PUBLIC_SCHOOLS
where `Elementary, Middle, or High School`='ES'

 * sqlite:///Chicago.db
Done.


COUNT(*)
462


In [ ]:
# 12. Display total number of elementary, middle and high school
# from Chicago_public_Schools
%%sql
select `Elementary, Middle, or High School`, COUNT(*) as `Total Schools`
from CHICAGO_PUBLIC_SCHOOLS
group by `Elementary, Middle, or High School`


 * sqlite:///Chicago.db
Done.


"Elementary, Middle, or High School",Total Schools
ES,462
HS,93
MS,11


In [ ]:
# 13. What is the highest Safety Score? #Which schools have highest Safety Score?
%%sql
select NAME_OF_SCHOOL, SAFETY_SCORE
from CHICAGO_PUBLIC_SCHOOLS
where SAFETY_SCORE=(
  select max(SAFETY_SCORE)
  from CHICAGO_PUBLIC_SCHOOLS
)

 * sqlite:///Chicago.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Abraham Lincoln Elementary School,99.0
Alexander Graham Bell Elementary School,99.0
Annie Keller Elementary Gifted Magnet School,99.0
Augustus H Burley Elementary School,99.0
Edgar Allan Poe Elementary Classical School,99.0
Edgebrook Elementary School,99.0
Ellen Mitchell Elementary School,99.0
James E McDade Elementary Classical School,99.0
James G Blaine Elementary School,99.0
LaSalle Elementary Language Academy,99.0
